## Step 4: Threat Model & Hyperparameters (30 clients, 3 attack types, 3 datasets)

Five malicious clients of thirty (16.7%):

| Client(s) | Attack type | Description |
|---|---|---|
| C00 | **Overt label-flip** | Multi-target: classes 0→1 AND 2→3, fraction=0.70, active from round 1 |
| C01, C02, C03 | **Slow label-flip** | Same multi-target flip, fraction ramps from 0% → 70% across rounds |
| C04 | **Gradient sign-inversion poisoning** | Sends `INVERSE_SCALE × local update` with `INVERSE_SCALE = −3` (sign flip + ×3 amplification; Byzantine-style, cf. Blanchard et al. 2017). *Not* a data-reconstruction ("gradient inversion") attack. |

**Hypotheses (not results):** distance-based rules such as Krum are expected to catch the large sign-inverted update; slow ramping is expected to be harder for single-round rules; CRTAT is designed to exploit temporal structure. Whether each expectation holds is decided only by the per-attack-type AUC table printed after the simulations.


In [ ]:
# ─── THESIS-OPTIMIZED v7 Threat Model ────────────────────────────
# 30 clients · 5 attackers (3 types) · 2 seeds · gradient sign-inversion attacker
# v7 NEW: CIFAR-100 as a third dataset, reconstruction-error Non-IID axis,
#         component-wise ablation config, and client-scale sweep config.

NUM_CLIENTS    = 30                # v5: was 10 — closer to realistic FL scale
NUM_POISONED   = 1                 # 1 overt label-flipper      (client 0)
NUM_SLOW       = 3                 # v5: was 1 — 3 slow attackers (clients 1,2,3)
NUM_INVERSE    = 1                 # v5 NEW: 1 gradient inverter (client 4)
NUM_MALICIOUS  = NUM_POISONED + NUM_SLOW + NUM_INVERSE   # 5 of 30 = 16.7%

# Label-flip target classes (multi-target attack)
POISON_FROM    = 0
POISON_TO      = 1
POISON_FROM_2  = 2
POISON_TO_2    = 3
POISON_FRAC    = 0.70              # 70% flip — strong attack
SLOW_RAMP_START= 2                 # slow attack begins at round 2

# Gradient sign-inversion poisoning scale
INVERSE_SCALE  = -3.0              # update sent = INVERSE_SCALE × local update → sign flip AND ×3 amplification (a pure sign flip would be -1.0)

FL_ROUNDS      = 15

# ─── Client participation per round ──────────────────────────────
CLIENT_FRACTION = 0.4              # v5: 12/30 per round (was 5/10)
                                   # Always-include-malicious rule relaxed
                                   # to avoid over-representing attackers.

# ─── Early stopping ──────────────────────────────────────────────
EARLY_STOP_PATIENCE  = 5
EARLY_STOP_DELTA     = 0.005
EARLY_STOP_MIN_ROUND = 10

# ─── Model / Feature Config ───────────────────────────────────────
CIFAR_PCA_DIM   = 512
CIFAR100_PCA_DIM= 512               # v7 NEW — same budget as CIFAR-10 for fair comparison
MNIST_PCA_DIM   = 256
FEATURE_DIM     = MNIST_PCA_DIM
HIDDEN_DIM      = 128
N_CLASSES       = 10                # default (CIFAR-10 / MNIST)
N_CLASSES_C100  = 100               # v7 NEW — CIFAR-100 fine-label classes
LR_INIT        = 0.08
LR_MIN         = 0.005
EPOCHS         = 3
BATCH_SIZE     = 128

# ─── CRTAT / Detection Config ─────────────────────────────────────
CRTAT_WINDOW   = 5
CRTAT_ALPHA    = 0.85               # weight on trend (slope)
CRTAT_BETA     = 0.2                # weight on volatility
CRTAT_DELTA    = 0.30               # weight on trend-change (difference of half-window slopes; NOT a time-normalised 2nd derivative)
CRTAT_GAMMA    = 0.88               # weight: temporal vs instantaneous
CRTAT_DECAY    = 0.90               # v6: trust-recovery factor per round
                                    # — exponentially decay old anomalies
                                    #   so honest clients with one bad round
                                    #   aren't permanently flagged
CRTAT_PENALTY_CAP = 0.80            # aggregation weight w_k ∝ exp(-3 · PENALTY_CAP · D_k)
                                    # soft down-weighting only; it does NOT bound influence
FLAG_THRESHOLD = 0.60               # v6: was 0.55 — tighter to reduce false positives

# ─── Ablation Study Config ───────────────────────────────────────
ABLATION_THRESHOLDS  = [0.50, 0.60, 0.70]
ABLATION_RAMP_STARTS = [2, 3, 5]

# Component ablation variants are defined once, in ABLATION_VARIANTS / EXTENSION_VARIANTS (Figure 15 / Table VI cell).

# v7 NEW: client counts swept for the Latency / Throughput / Storage vs
# Scale benchmarks (Result Analysis section).
SCALE_TEST_CLIENTS = [10, 30, 50, 100, 200, 500]

# ─── Reconstruction-Error Non-IID Config (v7 NEW) ────────────────
RECON_ERROR_BINS  = 10              # quantile bins used in place of class labels
RECON_ERROR_ALPHA = 0.5             # same Dirichlet concentration as the label split

# ─── v9 reviewer-response settings ───────────────────────────────
NEUTRAL_TRUST        = 1.0    # trust value inserted for rounds a client did NOT participate in (CRTAT(missing='neutral'), original behaviour)
USE_EARLY_STOPPING   = False  # stopping on mean-over-methods TEST accuracy is a test-dependent rule → off; every run uses all FL_ROUNDS
FINAL_K              = 3      # final client score = mean of the client's last FINAL_K OBSERVED (participating-round) scores — one rule for all methods/tables
METHOD_ORDER         = ['CRTAT+Blockchain','Krum','TrimmedMean','FoolsGold','CoordMedian','FedAvg_NoDefense']
ATTACK_UPDATE_SCALES = [1.0, 10.0, 50.0]   # malicious-update boost factors for the influence-bound experiment

# ─── Multi-seed (v5: was single seed) ────────────────────────────
SEEDS = [42, 123]
SEED  = SEEDS[0]                    # primary seed for non-aggregated cells
np.random.seed(SEED)

# ─── Client role assignments ─────────────────────────────────────
POISONED_CLIENTS = list(range(0, NUM_POISONED))                                    # [0]
SLOW_CLIENTS     = list(range(NUM_POISONED, NUM_POISONED + NUM_SLOW))              # [1,2,3]
INVERSE_CLIENTS  = list(range(NUM_POISONED + NUM_SLOW,
                              NUM_POISONED + NUM_SLOW + NUM_INVERSE))              # [4]
MALICIOUS        = POISONED_CLIENTS + SLOW_CLIENTS + INVERSE_CLIENTS               # [0,1,2,3,4]
HONEST_CLIENTS   = [i for i in range(NUM_CLIENTS) if i not in MALICIOUS]
gt_labels        = [1 if i in MALICIOUS else 0 for i in range(NUM_CLIENTS)]

# Per-attack-type ground truth (for split detection plots)
gt_overt   = [1 if i in POISONED_CLIENTS else 0 for i in range(NUM_CLIENTS)]
gt_slow    = [1 if i in SLOW_CLIENTS     else 0 for i in range(NUM_CLIENTS)]
gt_inverse = [1 if i in INVERSE_CLIENTS  else 0 for i in range(NUM_CLIENTS)]

print('Federated Learning — Threat Model (v7: 30 clients · 3 attack types · 3 datasets)')
print('═'*68)
print(f'  Clients total      : {NUM_CLIENTS}')
print(f'  Overt poisoned     : {NUM_POISONED}  → {POISONED_CLIENTS}')
print(f'  Slow attackers     : {NUM_SLOW}  → {SLOW_CLIENTS}')
print(f'  Sign-inversion     : {NUM_INVERSE}  → {INVERSE_CLIENTS}')
print(f'  Honest clients     : {len(HONEST_CLIENTS)}')
print(f'  Malicious ratio    : {NUM_MALICIOUS/NUM_CLIENTS*100:.1f}% ({NUM_MALICIOUS}/{NUM_CLIENTS})')
print(f'  FL rounds          : {FL_ROUNDS}')
print(f'  Local epochs       : {EPOCHS}')
print(f'  Client fraction    : {CLIENT_FRACTION:.0%} ({int(NUM_CLIENTS*CLIENT_FRACTION)} per round)')
print(f'  Attack (overt)     : multi-class label-flip ({POISON_FROM}→{POISON_TO} AND {POISON_FROM_2}→{POISON_TO_2}), fraction={POISON_FRAC}')
print(f'  Attack (slow)      : same flip, fraction ramps from rnd {SLOW_RAMP_START}→{FL_ROUNDS}')
print(f'  Attack (inverse)   : gradient sign-inversion, update × {INVERSE_SCALE} (sign flip + amplification)')
print(f'  Blockchain flag    : CRTAT score ≥ {FLAG_THRESHOLD}')
print(f'  Datasets           : CIFAR-10 (10cls), CIFAR-100 (100cls), MNIST (10cls)')
print(f'  Non-IID splits     : Dirichlet(label) + Reconstruction-Error(quality)')
print(f'  Random seeds       : {SEEDS}')
print(f'  Early stopping     : patience={EARLY_STOP_PATIENCE}, min_round={EARLY_STOP_MIN_ROUND}')


Federated Learning — Threat Model (v7: 30 clients · 3 attack types · 3 datasets)
════════════════════════════════════════════════════════════════════
  Clients total      : 30
  Overt poisoned     : 1  → [0]
  Slow attackers     : 3  → [1, 2, 3]
  Sign-inversion     : 1  → [4]
  Honest clients     : 25
  Malicious ratio    : 16.7% (5/30)
  FL rounds          : 15
  Local epochs       : 3
  Client fraction    : 40% (12 per round)
  Attack (overt)     : multi-class label-flip (0→1 AND 2→3), fraction=0.7
  Attack (slow)      : same flip, fraction ramps from rnd 2→15
  Attack (inverse)   : gradient sign-inversion, update × -3.0 (sign flip + amplification)
  Blockchain flag    : CRTAT score ≥ 0.6
  Datasets           : CIFAR-10 (10cls), CIFAR-100 (100cls), MNIST (10cls)
  Non-IID splits     : Dirichlet(label) + Reconstruction-Error(quality)
  Random seeds       : [42, 123]
  Early stopping     : patience=5, min_round=10
